# Lab 9: Named Entity Recognition (NER)

## Named Entity Recognition (NER)
Named Entity Recognition identifies and classifies entities such as people, organizations, locations, and dates in text.

1. Detect entities from the text.
2. Classify them into categories.

## NER using spaCy

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

text = "Apple Inc. was founded by Steve Jobs in Cupertino, California. It is known for its iPhone and Mac products."
doc = nlp(text)

print(f"Entities recognized in text : {doc.ents}")
for entity in doc.ents:
    print(f"Entity: {entity.text}, Label: {entity.label_}")

In [ ]:
for ent in doc.ents:
    print(f"Entity: {ent.text}")
    print(f"Label: {ent.label_}")
    print(f"Start Position: {ent.start}")
    print(f"End Position: {ent.end}")
    print(f"Entity Type: {spacy.explain(ent.label_)}")
    print("")

## Create a custom Named Entity Model
Entity annotations can be specified as character offsets and checked using BILUO tags.

In [ ]:
from spacy.training.iob_utils import offsets_to_biluo_tags

text = "Apple Inc. was founded by Steve Jobs in Cupertino, California. It is known for its iPhone and Mac products."
annotations = [
    (text.find("Apple Inc."), text.find("Apple Inc.") + len("Apple Inc."), "ORG"),
    (text.find("Steve Jobs"), text.find("Steve Jobs") + len("Steve Jobs"), "PERSON"),
    (text.find("Cupertino, California"), text.find("Cupertino, California") + len("Cupertino, California"), "GPE")
]
doc = nlp.make_doc(text)
tags = offsets_to_biluo_tags(doc, annotations)
print("Alignment Tags:", tags[:10])

## Task 1
1. Ask a user to enter a sentence, and identify the named entities in the sentence.
2. Print the details of each entity in the sentence provided by the user.

In [ ]:
sentence = input("Enter a sentence: ")
doc = nlp(sentence)

print(f"Entities recognized in text: {doc.ents}")
for ent in doc.ents:
    print(f"Entity: {ent.text}")
    print(f"Label: {ent.label_}")
    print(f"Start Position: {ent.start}")
    print(f"End Position: {ent.end}")
    print(f"Entity Type: {spacy.explain(ent.label_)}")
    print("")

## Task 2
Use the paragraph below:

“This agreement is made and entered into by and between ‘Abc & Co.’ and ‘Bcd LLC’ for term 1 year starting from April 1, 2020, hereinafter collectively referred to as the Parties. Samuel P. Jackson in the place (New York) and on the date written below, with the following terms and conditions.”

1. Find any entity that has not been identified and add it.
2. Find the count of `PERSON` and `ORG`.
3. Visualize the entities.

In [ ]:
from spacy.tokens import Span
from spacy.util import filter_spans
from spacy import displacy

text = "This agreement is made and entered into by and between ‘Abc & Co.’ and ‘Bcd LLC’ for term 1 year starting from April 1, 2020, hereinafter collectively referred to as the Parties. Samuel P. Jackson in the place (New York) and on the date written below, with the following terms and conditions."
doc = nlp(text)

print("Entities identified by spaCy:")
for ent in doc.ents:
    print(ent.text, ent.label_)

# Add entities that may have been missed by the pretrained model.
manual_entities = [("Abc & Co.", "ORG"), ("Bcd LLC", "ORG"), ("Samuel P. Jackson", "PERSON"), ("New York", "GPE"), ("April 1, 2020", "DATE")]
new_ents = list(doc.ents)
for name, label in manual_entities:
    start = text.find(name)
    if start != -1:
        span = doc.char_span(start, start + len(name), label=label, alignment_mode="expand")
        if span is not None and not any(span.start < ent.end and ent.start < span.end for ent in new_ents):
            new_ents.append(span)

doc.ents = filter_spans(new_ents)

print("\nEntities after adding missing entities:")
for ent in doc.ents:
    print(ent.text, ent.label_)

person_count = sum(ent.label_ == "PERSON" for ent in doc.ents)
org_count = sum(ent.label_ == "ORG" for ent in doc.ents)
print("PERSON count:", person_count)
print("ORG count:", org_count)

displacy.render(doc, style="ent", jupyter=True)

## Task 3: Training a Custom NER Model
1. Create a training example using `Example.from_dict()` with the provided text and annotations.
2. Train the NER model for 20 iterations.
3. Save the trained model in a folder named `custom_ner_model`.
4. Load the saved model and use it to identify named entities in a sentence.

In [ ]:
from spacy.training import Example
from spacy.util import minibatch
import random

text = "Microsoft was founded by Bill Gates and Paul Allen in Albuquerque, New Mexico. It is known for its Windows operating system and Office software products."

entities = []
for name, label in [("Microsoft", "ORG"), ("Bill Gates", "PERSON"), ("Paul Allen", "PERSON"), ("Albuquerque", "GPE"), ("New Mexico", "GPE"), ("Windows", "PRODUCT"), ("Office", "PRODUCT")]:
    start = text.find(name)
    entities.append((start, start + len(name), label))

# 1. Create a training example.
custom_nlp = spacy.blank("en")
ner = custom_nlp.add_pipe("ner")
for _, _, label in entities:
    ner.add_label(label)

example = Example.from_dict(custom_nlp.make_doc(text), {"entities": entities})
train_examples = [example]

# 2. Train the NER model for 20 iterations.
optimizer = custom_nlp.initialize(get_examples=lambda: train_examples)
for iteration in range(20):
    random.shuffle(train_examples)
    losses = {}
    for batch in minibatch(train_examples, size=1):
        custom_nlp.update(batch, sgd=optimizer, losses=losses)
    print(f"Iteration {iteration + 1}: {losses}")

# 3. Save the trained model.
custom_nlp.to_disk("custom_ner_model")

# 4. Load and test the saved model.
custom_nlp = spacy.load("custom_ner_model")
test_text = "Microsoft was founded by Bill Gates and Paul Allen in Albuquerque, New Mexico."
doc2 = custom_nlp(test_text)
for ent in doc2.ents:
    print(f"Entity: {ent.text}, Label: {ent.label_}")

displacy.render(doc2, style="ent", jupyter=True)